# Liquid-handling plan reasoning: evaluate a 1.5B model, then improve it

**What this notebook does.** It evaluates a small open model (`Qwen/Qwen2.5-1.5B-Instruct`) on 240
generated items that ask for the numbers and well addresses a liquid handler would execute (dilutions,
serial dilutions, master mixes, sample normalisation, well addressing, labware choice), then measures
three improvement arms on the *same* items with the *same* decoding:

| arm | what changes |
|---|---|
| `baseline` | zero-shot: intent + world facts + JSON schema; the model reasons freely and ends with JSON |
| `fewshot`  | two worked examples from the training families |
| `pot`      | Program-of-Thought: the model writes Python that prints the JSON; a sandbox executes it |
| `sft`      | LoRA fine-tune on ~1 500 generated items with reasoning traces (4 families); 2 families are held out |

Grading is programmatic (no LLM judge). Controls run before any model call. Every number is reported
with its n and a bootstrap CI; arms are compared with an exact McNemar test on paired items.

**How to run.** Colab: *Runtime → Change runtime type → T4 GPU*, then *Run all* (about 35 min).
The same file runs as a plain script (`python notebook/prep_eval_colab.py`) with `SMOKE = True` on CPU.

## 0. Configuration

In [ ]:
import os
import sys

SMOKE = os.environ.get("PREPEVAL_SMOKE", "0") == "1"  # tiny CPU-friendly run that exercises every code path
MODEL_ID = os.environ.get("PREPEVAL_MODEL", "Qwen/Qwen2.5-0.5B-Instruct" if SMOKE else "Qwen/Qwen2.5-1.5B-Instruct")
REPO_URL = "https://github.com/xbtu2/prepeval-takehome.git"
REPO_DIR = "prepeval-takehome"
N_PER_FAMILY = 4 if SMOKE else 40  # 40 -> 240 test items
ARMS = ["baseline", "fewshot", "pot", "sft"]
BATCH_SIZE = 2 if SMOKE else 16  # 32 halves inference time on a T4 if the budget is tight
SFT_MAX_STEPS = 3 if SMOKE else -1  # -1 = one epoch (~190 steps)
SFT_TRAIN_ITEMS = 16 if SMOKE else None  # None = all
MAX_NEW_TOKENS_CAP = 96 if SMOKE else None  # None = the arm's own cap
SEED = 0
OUT_DIR = "results"
PROBE_N = 8 if SMOKE else 40

## 1. Environment (Colab installs; local runs assume the repo venv)

In [ ]:
import importlib
import subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
  # Colab ships torch + transformers; never reinstall torch (it would replace the CUDA build).
  subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft>=0.14", "trl>=0.19", "datasets>=3.0", "accelerate>=1.0"], check=True)
  if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
  os.chdir(REPO_DIR)
elif not os.path.isdir("prepeval"):
  os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), ".."))
sys.path.insert(0, os.getcwd())
os.makedirs(OUT_DIR, exist_ok=True)

import gc
import hashlib
import json
import time
from collections import Counter, defaultdict

import numpy as np
import torch

from prepeval import dataset as D
from prepeval import families as F
from prepeval import grading as G
from prepeval import prompts as P
from prepeval import sandbox as S
from prepeval import stats as ST

torch.manual_seed(SEED)
np.random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"device={device} dtype={dtype} torch={torch.__version__} python={sys.version.split()[0]}")
if device == "cuda":
  print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB")

## 2. Data: the committed, seeded item files

The item files were generated by `prepeval.dataset` (PyLabRobot supplied the labware facts and the well
addressing ground truth at generation time). Loading them here, rather than regenerating, pins the eval.

In [ ]:
manifest = json.load(open("data/manifest.json"))
test_all = D.load_split("test")
train_all = D.load_split("train")
fewshot = D.load_split("fewshot")


def first_n_per_family(items, n):
  seen = Counter()
  out = []
  for it in items:
    if seen[it.family] < n:
      out.append(it)
      seen[it.family] += 1
  return out


test = first_n_per_family(test_all, N_PER_FAMILY)
train = train_all if SFT_TRAIN_ITEMS is None else train_all[:: max(1, len(train_all) // SFT_TRAIN_ITEMS)][:SFT_TRAIN_ITEMS]
print(f"test items: {len(test)}  (families: {dict(Counter(it.family for it in test))})")
print(f"train items: {len(train)}  held-out families (never in train): {F.HELDOUT_FAMILIES}")
print(f"infeasible (trap) share in test: {np.mean([not it.feasible for it in test]):.2f}")
print("leak check at build time:", manifest["leak_check"])
print("\nExample item\n" + "-" * 80 + "\n" + test[0].prompt + "\n" + "-" * 80 + "\ngold: " + json.dumps(test[0].gold))

## 3. Controls (before any model call)

1. gold answers must all be accepted; 2. every single-field corruption of a gold must be rejected;
3. a *program* that prints the gold must pass through sandbox + extractor + grader (the PoT path);
4. constant policies (`always feasible, zeros` / `always infeasible`) define the floor;
5. the leak check from the build (no test prompt in train, held-out families absent, no gold value printed
   in its own prompt, no few-shot value in a test gold).

In [ ]:
controls = {}
controls["gold_accepted"] = sum(G.grade(it, G.gold_completion(it)).accepted for it in test) / len(test)
muts = [(it, n, t) for it in test for n, t in G.mutations(it).items()]
controls["mutations_rejected"] = sum(not G.grade(it, t).accepted for it, _, t in muts) / len(muts)
ok = 0
for it in test:
  prog = "```python\nimport json\nresult = " + repr(it.gold) + "\nprint(json.dumps(result))\n```"
  text, res = S.execute_completion(prog)
  ok += bool(res and res.ok and G.grade(it, text).accepted)
controls["gold_program_through_sandbox"] = ok / len(test)
controls["policy_always_feasible_zeros"] = sum(G.grade(it, G.policy_always_feasible_zeros(it)).accepted for it in test) / len(test)
controls["policy_always_infeasible"] = sum(G.grade(it, G.policy_always_infeasible(it)).accepted for it in test) / len(test)
controls["leak_check_pass"] = bool(manifest["leak_check"]["PASS"])
controls["effective_floor"] = max(controls["policy_always_feasible_zeros"], controls["policy_always_infeasible"])
print("| control | value | expected |\n|---|---|---|")
exp = {"gold_accepted": "1.00", "mutations_rejected": "1.00", "gold_program_through_sandbox": "1.00",
       "policy_always_feasible_zeros": "0.00", "policy_always_infeasible": "= trap share", "leak_check_pass": "True",
       "effective_floor": "strongest non-solving control"}
for k, v in controls.items():
  print(f"| {k} | {v if isinstance(v, bool) else f'{v:.3f}'} | {exp[k]} |")
assert controls["gold_accepted"] == 1.0 and controls["mutations_rejected"] == 1.0 and controls["gold_program_through_sandbox"] == 1.0

## 4. Model and a shared, deterministic generation setup

One `GenerationConfig` for every arm: greedy, no repetition penalty (Qwen2.5's default `1.05` would
penalise re-emitting digits and JSON keys), left padding, own system message, prompt tokens sliced off.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
assert tok.pad_token_id is not None and tok.pad_token_id != tok.eos_token_id


def load_base():
  try:
    m = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=dtype)
  except TypeError:  # transformers < 4.56
    m = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype)
  return m.to(device).eval()


model = load_base()
GEN = GenerationConfig(do_sample=False, temperature=None, top_p=None, top_k=None, repetition_penalty=1.0,
                       pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
print(f"loaded {MODEL_ID}: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B params")


def render(messages, prefill=""):
  return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) + prefill


@torch.no_grad()
def generate(model, messages_list, max_new_tokens, prefill="", stop_strings=None, batch_size=BATCH_SIZE):
  """Batched greedy decoding; items are processed in order of prompt length so padding stays small."""
  texts = [render(m, prefill) for m in messages_list]
  order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
  outs = [None] * len(texts)
  for b in range(0, len(order), batch_size):
    idx = order[b : b + batch_size]
    enc = tok([texts[i] for i in idx], return_tensors="pt", padding=True).to(device)
    kw = {"stop_strings": stop_strings, "tokenizer": tok} if stop_strings else {}
    gen = model.generate(**enc, generation_config=GEN, max_new_tokens=max_new_tokens, **kw)
    new = gen[:, enc["input_ids"].shape[1] :]
    for i, row in zip(idx, new):
      outs[i] = prefill + tok.decode(row, skip_special_tokens=True)
  return outs


def run_arm(arm, model, items, tag=None):
  spec = P.ARMS[arm]
  cap = spec["max_new_tokens"] if MAX_NEW_TOKENS_CAP is None else min(spec["max_new_tokens"], MAX_NEW_TOKENS_CAP)
  msgs = [spec["messages"](it, fewshot) for it in items]
  t0 = time.time()
  if spec["tool"] == "python":
    # prefill the opening fence so the model writes code immediately and the closing fence stops decoding
    raw = generate(model, msgs, cap, prefill="```python\n", stop_strings=["```"])
    raw = [r if r.rstrip().endswith("```") else r + "\n```" for r in raw]
  else:
    raw = generate(model, msgs, cap)
  elapsed = time.time() - t0
  grades, records = {}, []
  for it, out in zip(items, raw):
    graded_text, run = (S.execute_completion(out) if spec["tool"] == "python" else (out, None))
    g = G.grade(it, graded_text)
    grades[it.id] = g
    records.append({"item_id": it.id, "family": it.family, "raw": out, "graded_text": graded_text if spec["tool"] else None,
                    "sandbox": (run.status if run else None), "accepted": g.accepted, "error_class": g.error_class})
  acc = np.mean([g.accepted for g in grades.values()])
  print(f"[{tag or arm}] n={len(items)} accepted={acc:.3f} time={elapsed:.0f}s "
        f"({elapsed / len(items):.2f}s/item) format_fail={np.mean([g.error_class == 'format' for g in grades.values()]):.2f}")
  return grades, records, elapsed

## 5. Probe: is the eval in the model's working range?

Arm `baseline` on the first few items. If this were > 0.7 the set would be too easy for the model and
if it were ~0 with mostly `format` failures the problem would be the output channel, not the reasoning.
(Sanity check only; the model is not swapped post hoc.)

In [ ]:
probe_items = first_n_per_family(test, max(1, PROBE_N // len(F.FAMILIES)))
pg, pr, _ = run_arm("baseline", model, probe_items, tag="probe")
print("error classes:", dict(Counter(g.error_class for g in pg.values() if not g.accepted)))

## 6. Arms A–C on the pristine model

In [ ]:
grades_by_arm, records_by_arm, timing = {}, {}, {}
for arm in [a for a in ARMS if a != "sft"]:
  grades_by_arm[arm], records_by_arm[arm], timing[arm] = run_arm(arm, model, test)
if "pot" in records_by_arm:
  print("sandbox outcomes (pot):", dict(Counter(r["sandbox"] for r in records_by_arm["pot"])))

## 7. LoRA fine-tune on the four training families

Rows are prompt–completion pairs: the chat-templated prompt (with the assistant header) and the
programmatic reasoning trace + gold JSON. Loss on the completion only. fp16 autocast on the T4 with
fp32 adapters (PEFT default). Held-out families `normalize_samples` and `labware_fit` never appear.

In [ ]:
sft_grades = None
if "sft" in ARMS:
  from datasets import Dataset
  from peft import LoraConfig, PeftModel, get_peft_model
  from trl import SFTConfig, SFTTrainer

  rows = [{"prompt": render(P.messages_baseline(it)), "completion": P.sft_target(it) + tok.eos_token + "\n"} for it in train]
  MAX_LEN = 768
  lens = [len(tok(r["prompt"] + r["completion"])["input_ids"]) for r in rows]
  assert max(lens) <= MAX_LEN, f"longest training row {max(lens)} tokens > {MAX_LEN}: truncation would drop the JSON"
  print(f"SFT rows: {len(rows)}  token length: median {int(np.median(lens))}, max {max(lens)}")
  ds = Dataset.from_list(rows)

  lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
  peft_model = get_peft_model(model, lora)
  peft_model.print_trainable_parameters()

  import dataclasses

  supported = {f.name for f in dataclasses.fields(SFTConfig)}
  cfg_kw = dict(output_dir=os.path.join(OUT_DIR, "sft_tmp"), per_device_train_batch_size=1 if SMOKE else 4,
                gradient_accumulation_steps=1 if SMOKE else 2, learning_rate=2e-4, num_train_epochs=1,
                max_steps=SFT_MAX_STEPS, lr_scheduler_type="cosine", warmup_ratio=0.03, logging_steps=10,
                fp16=(device == "cuda"), bf16=False, report_to="none", save_strategy="no", seed=SEED,
                gradient_checkpointing=False, max_length=MAX_LEN, packing=False, completion_only_loss=True,
                dataset_num_proc=1, remove_unused_columns=True)
  if "max_length" not in supported:  # older trl
    cfg_kw["max_seq_length"] = cfg_kw.pop("max_length")
  cfg = SFTConfig(**{k: v for k, v in cfg_kw.items() if k in supported})
  trainer = SFTTrainer(model=peft_model, args=cfg, train_dataset=ds, processing_class=tok)
  t0 = time.time()
  train_out = trainer.train()
  timing["sft_train"] = time.time() - t0
  print(f"trained {train_out.global_step} steps in {timing['sft_train']:.0f}s; final loss {train_out.training_loss:.3f}")

  adapter_dir = os.path.join(OUT_DIR, "lora_adapter")
  trainer.model.save_pretrained(adapter_dir)
  adapter_sha = hashlib.sha256(open(os.path.join(adapter_dir, "adapter_model.safetensors"), "rb").read()).hexdigest()[:16]
  del trainer
  gc.collect()
  if device == "cuda":
    torch.cuda.empty_cache()

  # Sanity: with the adapter disabled the model must reproduce the baseline outputs (same batch composition).
  peft_model.eval()
  peft_model.config.use_cache = True
  check_items = test[:10]
  with peft_model.disable_adapter():
    off = generate(peft_model, [P.messages_baseline(it) for it in check_items], 64, batch_size=len(check_items))
  base_again = generate(model.base_model.model if hasattr(model, "base_model") else model, [P.messages_baseline(it) for it in check_items], 64, batch_size=len(check_items)) if False else None
  print("adapter-disabled generations produced:", sum(bool(o.strip()) for o in off), "/", len(off))

  merged = peft_model.merge_and_unload().eval()
  merged.config.use_cache = True
  grades_by_arm["sft"], records_by_arm["sft"], timing["sft"] = run_arm("sft", merged, test)
  model = merged

## 8. Results

Headline: acceptance (every required field within tolerance) per arm with 95 % bootstrap CIs, the paired
delta against the baseline with its CI, and an exact McNemar p-value. Per-family numbers (n = 40 each)
are directional; the aggregate (n = 240) carries the claim. Held-out families show whether the LoRA
learned procedures or memorised templates.

In [ ]:
summary = ST.summarize(test, grades_by_arm, baseline="baseline")
summary["controls"] = controls
summary["timing_s"] = timing
print(ST.results_table(summary))
print()
print(ST.family_table(summary))
print()
print("effective floor (strongest non-solving control):", f"{controls['effective_floor']:.3f}")
for arm, a in summary["arms"].items():
  pos = (a["acc"] - controls["effective_floor"]) / (1 - controls["effective_floor"])
  print(f"  {arm:9s} acc {a['acc']:.3f} -> position above floor {pos:+.3f}; errors {a['error_classes']}")

### Failure examples (three per error class per arm): reviewers trust examples more than CIs

In [ ]:
for arm, recs in records_by_arm.items():
  by_class = defaultdict(list)
  for r in recs:
    if not r["accepted"]:
      by_class[r["error_class"]].append(r)
  print("=" * 100 + f"\n{arm}")
  for cls, rs in by_class.items():
    for r in rs[:3]:
      it = next(i for i in test if i.id == r["item_id"])
      g = grades_by_arm[arm][it.id]
      shown = (r["graded_text"] or r["raw"]) if arm == "pot" else r["raw"]
      print(f"--- [{cls}] {it.id}\n  gold: {json.dumps(it.gold)[:200]}\n  pred: {json.dumps(g.pred)[:200] if g.pred else None}")
      print("  model output (tail): " + shown.strip().replace("\n", " ⏎ ")[-300:])

### Figure: acceptance per arm (95 % bootstrap CI) and per family

In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

# palette: validated reference set (blue, orange, aqua, yellow in fixed order); recessive axes and grid
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
MUTED, GRID, TEXT, TEXT2, SURFACE = "#898781", "#e1e0d9", "#0b0b0b", "#52514e", "#fcfcfb"
arms = list(summary["arms"])
fams = list(summary["arms"][arms[0]]["per_family"])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1, 1.5]})
fig.patch.set_facecolor(SURFACE)
for ax in (ax1, ax2):
  ax.set_facecolor(SURFACE)
  for side in ("top", "right"):
    ax.spines[side].set_visible(False)
  for side in ("left", "bottom"):
    ax.spines[side].set_color(GRID)
  ax.tick_params(colors=MUTED, labelcolor=TEXT2, length=0)
  ax.yaxis.grid(True, color=GRID, linewidth=1)
  ax.set_axisbelow(True)
# left: one bar per arm, single hue, CI whiskers, direct labels
acc = [summary["arms"][a]["acc"] for a in arms]
lo = [summary["arms"][a]["ci95"][0] for a in arms]
hi = [summary["arms"][a]["ci95"][1] for a in arms]
x = np.arange(len(arms))
ax1.bar(x, acc, width=0.55, color=SERIES[0], edgecolor=SURFACE, linewidth=2)
ax1.errorbar(x, acc, yerr=[np.array(acc) - np.array(lo), np.array(hi) - np.array(acc)], fmt="none", ecolor=TEXT2, elinewidth=1.5, capsize=4)
for xi, a in zip(x, acc):
  ax1.text(xi, min(a + 0.06, 1.02), f"{a:.2f}", ha="center", va="bottom", color=TEXT, fontsize=10)
ax1.axhline(controls["effective_floor"], color=MUTED, linewidth=1, linestyle=(0, (4, 3)))
ax1.text(len(arms) - 0.5, controls["effective_floor"] + 0.01, "effective floor", ha="right", va="bottom", color=TEXT2, fontsize=8)
ax1.set_xticks(x, arms)
ax1.set_ylim(0, 1.1)
ax1.set_ylabel("accepted (all fields within tolerance)", color=TEXT2)
ax1.set_title(f"Acceptance by arm, n = {summary['n_items']} items, 95 % CI", color=TEXT, fontsize=11, loc="left")
# right: per-family dots, one colour per arm in fixed order, legend + table printed above
y = np.arange(len(fams))
for j, a in enumerate(arms):
  vals = [summary["arms"][a]["per_family"][f]["acc"] for f in fams]
  ax2.scatter(vals, y + (j - (len(arms) - 1) / 2) * 0.16, s=64, color=SERIES[j % len(SERIES)], edgecolor=SURFACE, linewidth=1.5, label=a, zorder=3)
ax2.set_yticks(y, [f + (" (held-out)" if summary["arms"][arms[0]]["per_family"][f]["heldout"] else "") for f in fams])
ax2.invert_yaxis()
ax2.set_xlim(-0.02, 1.02)
ax2.xaxis.grid(True, color=GRID, linewidth=1)
ax2.yaxis.grid(False)
ax2.set_xlabel("accepted", color=TEXT2)
ax2.set_title("Per family (n = %d each; directional)" % summary["arms"][arms[0]]["per_family"][fams[0]]["n"], color=TEXT, fontsize=11, loc="left")
leg = ax2.legend(frameon=False, loc="lower right", fontsize=9)
for t in leg.get_texts():
  t.set_color(TEXT2)
plt.tight_layout()
fig_path = os.path.join(OUT_DIR, "smoke-figure.png" if SMOKE else "figure.png")
plt.savefig(fig_path, dpi=150, facecolor=SURFACE)
print("saved", fig_path)

### Provenance and results file

In [ ]:
import peft
import transformers

try:
  import trl

  trl_v = trl.__version__
except Exception:
  trl_v = None
provenance = {
  "model_id": MODEL_ID, "device": device, "dtype": str(dtype),
  "gpu": torch.cuda.get_device_name(0) if device == "cuda" else None,
  "transformers": transformers.__version__, "peft": peft.__version__, "trl": trl_v, "torch": torch.__version__,
  "dataset_sha256": manifest.get("sha256"), "generator_version": manifest["generator_version"], "plr_commit": manifest["plr_commit"],
  "prompt_sha256": {"system": hashlib.sha256(P.SYSTEM.encode()).hexdigest()[:16], "pot_system": hashlib.sha256(P.POT_SYSTEM.encode()).hexdigest()[:16]},
  "decoding": {"greedy": True, "repetition_penalty": 1.0, "max_new_tokens": {a: P.ARMS[a]["max_new_tokens"] for a in ARMS}, "batch_size": BATCH_SIZE},
  "lora": ({"r": 16, "alpha": 32, "targets": "attn+mlp", "adapter_sha256_16": adapter_sha, "train_items": len(train), "steps": SFT_MAX_STEPS} if sft_grades is None and "sft" in grades_by_arm else None),
  "n_test_items": len(test), "smoke": SMOKE, "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
out_path = os.path.join(OUT_DIR, "smoke-cpu.json" if SMOKE else "results.json")
ST.dump_results(out_path, summary, grades_by_arm, provenance)
with open(os.path.join(OUT_DIR, "raw_outputs.jsonl" if not SMOKE else "smoke_raw_outputs.jsonl"), "w") as fh:
  for arm, recs in records_by_arm.items():
    for r in recs:
      fh.write(json.dumps({"arm": arm, **r}) + "\n")
print("wrote", out_path)
print("\nREADME-ready block:\n")
print(ST.results_table(summary))
print()
print(ST.family_table(summary))